# FFIQ — Indian-language voice PILOT (IndicF5)

Clones the scripts in `FFIQ_indic/input/*.json` with **IndicF5** (AI4Bharat, MIT; Hindi, Marathi, Tamil, Kannada, Telugu…).
Fully separate from the English course: it only READS `FFIQ_voice/voice_reference_A.wav` and never writes to `FFIQ_voice/`.

**One-time setup (IndicF5 is gated):**
1. Sign in at huggingface.co and click **Agree and access** on https://huggingface.co/ai4bharat/IndicF5
2. Make a **Read** token at https://huggingface.co/settings/tokens
3. In Colab, click the **key icon (Secrets)** on the left → add `HF_TOKEN` = your token → switch on **Notebook access**.

Then: **Runtime → Change runtime type → T4 GPU**, and **Runtime → Run all**.
The first time, Cell 1 installs IndicF5 and restarts Colab by itself ("session crashed" is expected) — then click **Runtime → Run all** once more.
Results land in `FFIQ_indic/out/`.

In [ ]:
# CELL 1 — Install IndicF5 in ONE pip call (~3-5 min), then restart once
# Installing swaps numpy for the version IndicF5 needs, and Colab must restart
# to load it. This cell does that by itself: after the restart, click
# Runtime -> Run all AGAIN; this cell then sees IndicF5 is installed and skips.
import os, subprocess, sys
MARK = '/content/indicf5_installed.txt'
if os.path.exists(MARK):
    from importlib.metadata import version
    for p in ('torch', 'transformers', 'numpy'):
        try:
            print(f'{p:13}', version(p))
        except Exception:
            print(f'{p:13}', 'not installed')
    print('✅ IndicF5 already installed - continuing.')
else:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/ai4bharat/IndicF5.git'],
                       capture_output=True, text=True)
    print(r.stdout[-1500:]); print(r.stderr[-3000:])
    if r.returncode:
        raise RuntimeError('STOP: IndicF5 did not install. Paste this whole output to Claude.')
    open(MARK, 'w').write('ok')
    print('✅ IndicF5 installed. Restarting now - when Colab reconnects, click Runtime -> Run all again.')
    os.kill(os.getpid(), 9)   # restart the Python session so the new numpy loads

In [ ]:
# CELL 2 — Hugging Face access (needs the HF_TOKEN secret, see the notes at the top)
from google.colab import userdata
from huggingface_hub import login
try:
    login(userdata.get('HF_TOKEN'))
except Exception as e:
    raise RuntimeError('STOP: add your Hugging Face token as the Colab secret HF_TOKEN '
                       '(key icon on the left, Notebook access ON). ' + repr(e))
print('✅ Hugging Face connected.')

In [ ]:
# CELL 3 — Google Drive + load IndicF5
from google.colab import drive
drive.mount('/content/drive', force_remount=True)   # fresh view of files your PC just added
import os, glob, json, hashlib, time, datetime, random
import numpy as np, soundfile as sf, torch
from transformers import AutoModel

INDIC = '/content/drive/MyDrive/FFIQ_indic'      # this project's own folder
PILOT = f'{INDIC}/input'
VOICE = '/content/drive/MyDrive/FFIQ_voice/voice_reference_A.wav'   # approved voice, read only
# exact words spoken in voice_reference_A.wav (IndicF5 needs the transcript).
# Kept on Drive, not in this notebook: course content stays off GitHub.
REF_TXT = f'{INDIC}/voice_reference_A.txt'
if not os.path.exists(REF_TXT):
    raise RuntimeError('STOP: FFIQ_indic/voice_reference_A.txt missing (ask Claude to put it back).')
REF_TEXT = open(REF_TXT, encoding='utf-8').read().strip()
JOBS = sorted(glob.glob(f'{PILOT}/*.json'))
print('Pilot scripts:', [os.path.basename(j) for j in JOBS] or 'none')
if not JOBS:
    raise RuntimeError('STOP: no .json in FFIQ_indic/input/')
if not os.path.exists(VOICE):
    raise RuntimeError('STOP: voice_reference_A.wav missing in FFIQ_voice/')

model = AutoModel.from_pretrained('ai4bharat/IndicF5', trust_remote_code=True)
if torch.cuda.is_available():
    try:
        model = model.to('cuda')
    except Exception as e:
        print('note: model manages its own device:', e)
print('✅ IndicF5 loaded. GPU:', torch.cuda.is_available())

In [ ]:
# CELL 4 — Clone the pilot clips (resumes: finished clips are kept)
SEED = 1019   # same seed as the English voice: same text -> same delivery

def speak(text, ref_wav, ref_text, out):
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    audio = np.asarray(model(text, ref_audio_path=ref_wav, ref_text=ref_text))
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    sf.write(out, audio.astype(np.float32), 24000)

# Voice A always. If you add FFIQ_indic/input/voice_reference_hi.wav + .txt (10-15 s of you
# speaking Hindi + its exact words), a second set is made to compare the accent.
refs = [('voiceA', VOICE, REF_TEXT)]
hi_wav, hi_txt = f'{PILOT}/voice_reference_hi.wav', f'{PILOT}/voice_reference_hi.txt'
if os.path.exists(hi_wav) and os.path.exists(hi_txt):
    refs.append(('hindi_ref', hi_wav, open(hi_txt, encoding='utf-8').read().strip()))

for job in JOBS:
    name = os.path.basename(job)[:-5]
    scripts = json.load(open(job, encoding='utf-8'))
    for tag, wav, txt in refs:
        out_dir = f'{INDIC}/out/{name}_{tag}'
        os.makedirs(out_dir, exist_ok=True)
        print(f'\n=== {name} / {tag}: {len(scripts)} clips ===')
        for key, text in scripts.items():
            out = f'{out_dir}/{key}.wav'
            if os.path.exists(out):
                print('  have', key); continue
            t0 = time.time()
            speak(text, wav, txt, out)
            print(f'  {key}  {time.time() - t0:.0f}s')
        json.dump({'lesson': name, 'engine': 'indicf5', 'model': 'ai4bharat/IndicF5',
                   'reference': os.path.basename(wav),
                   'voice_md5': hashlib.md5(open(wav, 'rb').read()).hexdigest(),
                   'seed': SEED, 'keys': list(scripts),
                   'finished': datetime.datetime.now().isoformat(timespec='seconds')},
                  open(f'{out_dir}/_DONE.json', 'w'), indent=2)

drive.flush_and_unmount()
print('\n✅ Clips saved to Drive: FFIQ_indic/out/  (tell Claude)')